# Missing Rate Analysis

Analyze and visualize missing data rates in extracted PVIGR datasets.

This notebook generates heatmaps showing missing data patterns across sensors for:
- Power data (between 6:00-18:00)
- Microclimate data
- Weather station data
- Soil moisture data

The analysis automatically discovers all data types in the extracted directory.

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path
import warnings
import importlib
warnings.filterwarnings('ignore')

# Set project root relative to notebook location
project_root = Path.cwd().parent

# Add src to path
sys.path.insert(0, str(project_root / 'src'))

# Import and reload missing_rate_analysis module
import missing_rate_analysis
importlib.reload(missing_rate_analysis)
from missing_rate_analysis import analyze_all_folders

## Configuration

Define the paths for input data and output figures.

In [ ]:
# Define paths (allow override via ANALYSIS_PROCESSED_ROOT)
analysis_root_env = os.environ.get('ANALYSIS_PROCESSED_ROOT')
if analysis_root_env:
    analysis_root = Path(analysis_root_env)
else:
    analysis_root = project_root / 'processed_data'
    # Find most recent timestamped folder
    candidate_dirs = sorted(
        [p for p in analysis_root.iterdir() if p.is_dir() and p.name[0].isdigit() and '_' in p.name],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if candidate_dirs:
        analysis_root = candidate_dirs[0]

extracted_dir = analysis_root / 'extracted'
output_dir = analysis_root / 'missing_rate'

print(f'Analyzing data from: {extracted_dir}')
print(f'Output directory: {output_dir}')

## Run Analysis

Process all folders and generate heatmaps for each data type.

In [ ]:
# Run the analysis
results = analyze_all_folders(extracted_dir, output_dir)

## Summary

In [ ]:
# Summary: Count plots generated by task
if results['successful']:
    success_df = pd.DataFrame(results['successful'])
    task_summary = success_df.groupby('data_type').size()
    
    print(f"\nGenerated {len(success_df)} heatmaps:")
    for task, count in task_summary.items():
        print(f"  {task.replace('_', ' ').title()}: {count} plot(s)")
    
    if results['failed']:
        print(f"\nFailed: {len(results['failed'])} analysis(es)")
else:
    print("No analyses completed.")

In [ ]:
if results['successful']:
    print(f"\nAll heatmaps saved to: {output_dir}")